# CSA prompt pilot — CONTROL,500updates,T4×2
**Một notebook/một tài khoản, mỗi notebook2GPU.** Select **GPU T4×2**, Internet ON, Save Version → Save & Run All. Không cần Secret/API.
Gắn3inputDatasets: `csa-prompt-pilot-inputs-v169` (cache+161weights, cần share quyền cho Ngân), `csa-full-roi-karpathy-v134`, `vuthetam/mscoco-2014` (dataset_coco.json).
Chỉnh path ở cell đầu nếu Kaggle mount khác; không đổi SHA/nguồn. Không cần dataset detector/UPT/JPEG/bank130 riêng; banks nằm trong161.

Giữ noun/verb banks. CONTROL tắt prompt adapter và chỉ học CSA.
Cùng exact161/freshAdam5e-6/CE+.05SCST/FP32/dropoutOFF/clip1. TRAIN pool10k,500×8=4000uniqueimageevents, không phải full epoch.
Actual161/CUDA/DDP guards trước train;2smokeupdates bị bỏ,worker nạp161 lại. Parent3h STOPcap,không hứa ETA/billing; checkpoint100updates, không tự restart.
ORIGINAL+END fullVALID5000 beam5/max20 và fullrefNLL/CIDEr/B4/ROUGE/CHAIR/recall; PROMPT thêm64image sensitivity probe. KhôngTEST,không chọn checkpoint theoVALID.
Lưu5beams/ảnh cho **bước rerank cố định tiếp theo**, notebook này chỉ đo rank0; chưa có điểm sau rerank. Giữ checkpoint.bin/wholeoutput và gửi ZIPreview.
Tác giả chỉ kiểm metadata/NumPy/AST/sourcebundle; thực thi model/DDP/API trên máy tác giả không được thực hiện.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='CONTROL'

WALL_HOURS=3 # Parent STOPcap, not ETA or Kaggle quota/billing promise.
INPUT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-pilot-inputs-v169')
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
OUTPUT=Path('/kaggle/working/csa_prompt_pilot_'+ARM+'_500_v1')
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
for p in (INPUT_ROOT,FULL_ROOT,COCO_JSON):
    if not p.exists():raise FileNotFoundError('Attach3inputDatasets BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use fresh runtime/output path. No automatic restart.')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for fullROI stage and atomic checkpoints')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-'+ARM.lower()+'-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
REVISION='fc54bdea66479e66c8e7662831be167325ffef8e'
MAPPING={'experiments/csa_paper/prompt_pilot_data.py': 'prompt_pilot_data.py', 'experiments/csa_paper/prompt_pilot_model.py': 'prompt_pilot_model.py', 'experiments/csa_paper/run_prompt_pilot.py': 'run_prompt_pilot.py', 'experiments/csa_paper/test_prompt_pilot_data.py': 'test_prompt_pilot_data.py', 'experiments/csa_paper/make_prompt_pilot_notebooks.py': 'make_prompt_pilot_notebooks.py', 'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/test_prompt_adapter_model.py': 'test_prompt_adapter_model.py', 'experiments/csa_paper/prompt_text_cache.py': 'prompt_text_cache.py', 'experiments/csa_paper/scst_pilot_data.py': 'scst_pilot_data.py', 'experiments/csa_paper/scst_pilot_model.py': 'scst_pilot_model.py', 'experiments/csa_paper/run_scst_pilot.py': 'run_scst_pilot.py', 'experiments/csa_paper/test_scst_pilot_model.py': 'test_scst_pilot_model.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/full_xe_data.py': 'full_xe_data.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/PROMPT_PILOT_PROTOCOL.md': 'PROMPT_PILOT_PROTOCOL.md', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/scst.py': 'captioning/scst.py'}
HASHES={'experiments/csa_paper/prompt_pilot_data.py': '28515b2c98c056d9093f82fbd51ee0a24819e98467015753f294378996c088f9', 'experiments/csa_paper/prompt_pilot_model.py': 'e47c869f7ef74af6ab734b2ac5de0973a53aae0234c2f56141a432de1d7a430c', 'experiments/csa_paper/run_prompt_pilot.py': 'cf1add9661c8ebf687da989d03c1d8780a8176bb1b4262e81203fa4c1268b347', 'experiments/csa_paper/test_prompt_pilot_data.py': 'bd9a883aabeb6ff6e2bcc4a4fc7644d48fa090f85c8cc2fe08b7a7569b8d88f5', 'experiments/csa_paper/make_prompt_pilot_notebooks.py': 'b7cf7d9cf18f99785f51337108c34eee9f18e7415e677b4932024d43d09bedf7', 'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/test_prompt_adapter_model.py': '152d7419238d9484c5b16e84fd987f3e9f68a57649f17019e5532860e144554a', 'experiments/csa_paper/prompt_text_cache.py': '1072c0111d1df213a1c3e9054c1db57d0746bdbdfb639caa0c64736da7d7ffe6', 'experiments/csa_paper/scst_pilot_data.py': '119324b3933e07a07ff5aa60a4cb4272ebfdd587d0451e56f338296954a2743b', 'experiments/csa_paper/scst_pilot_model.py': '0b5e2d384ffc7195c82888f3f7904eac145ba7291ebed27dc8342ed8e1ac9576', 'experiments/csa_paper/run_scst_pilot.py': '9a640766bd063d2e5d5854dab9177168ae6528bf0fcd43467ba7e5b38618285e', 'experiments/csa_paper/test_scst_pilot_model.py': 'd1192a844ad111f3423d39f6cdece1daca88417b3255bd44fd1f096bc9588ed5', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/PROMPT_PILOT_PROTOCOL.md': '7b64cb87199dcc4abab951828904465146356cceafcd059fa9a387688102122f', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e'}

for original,destination in MAPPING.items():
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during source download')
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original,timeout=min(90,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[original]:raise RuntimeError('Changed pinned source:'+original)
    target=SOURCE/destination;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTHONPATH']=str(SOURCE)
CHAIR=SOURCE/'chair';CONFIG=SOURCE/'config.json'
cfg=dict(arm=ARM,updates=500,input_root=str(INPUT_ROOT),full_root=str(FULL_ROOT),dataset_json=str(COCO_JSON),
    output=str(OUTPUT),stage=str(SOURCE/'stage'),chair=str(CHAIR),worker_seconds=6000)
CONFIG.write_text(json.dumps(cfg),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,max(.01,PARENT_DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Prompt pilot active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Prompt pilot phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_pilot_data.py'],timeout=120)
    # Fail weight/cache identities before slow fullROI staging or GPU smoke.
    sys.path.insert(0,str(SOURCE))
    from prompt_pilot_data import checkpoint,cache_root
    print('Exact161 checkpoint:',checkpoint(INPUT_ROOT),flush=True)
    print('Exact169 cache:',cache_root(INPUT_ROOT)[3],flush=True)
    probe="import json,torch;print(json.dumps(dict(version=torch.__version__,available=torch.cuda.is_available(),count=torch.cuda.device_count())));assert torch.cuda.is_available() and torch.cuda.device_count()==2,'Select GPU T4x2 and restart; do not reinstall Torch blindly'"
    run([sys.executable,'-c',probe],timeout=60)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2','nltk==3.9.2','future'],timeout=300)
    run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/clips/pattern.git@af754685cca3713db0abc4f020f2e94467c19d85'],timeout=300)
    run([sys.executable,'-c',"import nltk;from pattern.en import singularize;assert singularize('umbrellas')=='umbrella';assert nltk.download('punkt');assert nltk.download('punkt_tab')"],timeout=300)
    run(['java','-version'],timeout=30)
    run(['git','init',CHAIR],timeout=60)
    run(['git','-C',CHAIR,'remote','add','origin','https://github.com/Maxlinn/CHAIR-metric-standalone.git'],timeout=60)
    run(['git','-C',CHAIR,'fetch','--depth','1','origin','4087a26211aa2339b9a76307cb8f0321ef691d0a'],timeout=300)
    run(['git','-C',CHAIR,'checkout','--detach','FETCH_HEAD'],timeout=60)
    run([sys.executable,'-u',SOURCE/'run_prompt_pilot.py','prepare','--config',CONFIG],timeout=1800)
    run([sys.executable,'-u',SOURCE/'run_prompt_pilot.py','metric_preflight','--config',CONFIG],timeout=240)
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_model.py'],timeout=180)
    launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
    run([*launch,SOURCE/'test_scst_pilot_model.py','--ddp'],timeout=240)
    run([*launch,SOURCE/'run_prompt_pilot.py','preflight','--config',CONFIG],timeout=600)
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<2400:raise RuntimeError('Too little parent budget after staging/smoke; preserve stage/log, do not train automatically')
    cfg['worker_seconds']=remaining-900;CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
    (OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2),encoding='utf-8')
    run([*launch,SOURCE/'run_prompt_pilot.py','worker','--config',CONFIG],timeout=cfg['worker_seconds']+30)
    run([sys.executable,'-u',SOURCE/'run_prompt_pilot.py','score','--config',CONFIG],timeout=850)
    complete=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))['status']=='PROMPT_PILOT500_COMPLETE'
    if not complete:raise RuntimeError('Incomplete endpoint/VALID, no partial ranking')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,updates_requested=500,test_used=False,rerank_measured=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.jsonl','.log','.txt') and p.name!='training_references.json':z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',REVIEW,'\nKEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('KEEP CHECKPOINT:',OUTPUT/'checkpoint.bin' if (OUTPUT/'checkpoint.bin').is_file() else 'No checkpoint yet',flush=True)
    print('COMPLETE500 + original/END VALID5000.5beams saved; rerank is the next scoring step.' if complete else 'INCOMPLETE; preserve output; no automatic restart.',flush=True)
